# B5T2 · Cuaderno de mando de la evaluación

Este cuaderno **conduce**; el código **vive en el paquete `agente/`**. La separación no es cosmética:

- El día 24 un evaluador clona el repo y ejecuta `evaluar("holdout.jsonl")` sin abrir ningún notebook. Si la lógica estuviera aquí, no funcionaría. Ese fue exactamente el problema del repo del 17-sep.
- Todo lo que se ve aquí se puede volver a generar con `python -m agente.cli …`. Este cuaderno es la misma máquina con ventanas.

Qué hace cada bloque:

| Bloque | Gasta API | Para qué |
|---|---|---|
| 1 · Preparación y **calentamiento** | no | comprobar corpus y clave, precargar el retrieval |
| 2 · Una pregunta | sí (1) | ver una trayectoria completa antes de lanzar 60 |
| 3 · Ejecutar una arquitectura | sí (20 × reps) | guarda un JSON por pregunta; **idempotente** |
| 4 · Leer la tabla | no | qué falla, y por qué, mirando la trayectoria guardada |
| 5 · Comparativa | no | la tabla del informe |
| 6 · Recall del retrieval | casi no | la tabla del §4.4 |

Cambia `ARQ` y vuelve a ejecutar los bloques 3-5 para cada peldaño:

`baseline` → `a1_guardrails` → `a2_retrieval` → `a3_hibrido` → `a4_comparativas`

Cada peldaño **añade** al anterior y no quita nada, así la diferencia entre dos filas consecutivas se atribuye a una sola cosa.

## 1 · Preparación

**Por qué hay un calentamiento.** La primera vez que el agente llama a `search_filings`, Python carga el índice FAISS y el modelo de embeddings desde disco: entre 15 y 20 segundos. Si eso ocurre *dentro* de una pregunta, esos segundos se suman a su latencia y la columna del informe queda contaminada.

Y hay algo peor: el proceso del kernel sobrevive entre repeticiones, así que **solo la repetición 1 paga la carga**. En la tabla eso aparece como varianza entre repeticiones — precisamente lo que las tres repeticiones intentan medir. Un artefacto de arranque disfrazado de ruido del modelo.

`calentar()` lo saca fuera del cronómetro. `ejecutar()` la llama sola, pero conviene verla aquí para saber qué está pasando.

De paso, si el modelo ya está en la caché local, pone `HF_HUB_OFFLINE=1`: desaparece el aviso de peticiones anónimas a Hugging Face, se ahorra un viaje de red por arranque y —lo que de verdad importa— **el día 24 la ejecución no depende de que Hugging Face esté disponible**. En un clon recién hecho, sin caché, no se activa, para que la primera descarga funcione con normalidad.

In [ ]:
# 1 · Preparación ---------------------------------------------------------------
# autoreload: si editas un fichero de agente/, el cuaderno lo recoge sin reiniciar.
%load_ext autoreload
%autoreload 2

import os, json, warnings
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# tqdm quiere dibujar una barra de progreso interactiva y necesita `ipywidgets`.
# Sin él se cae a la barra de texto, que funciona igual. Silenciamos el aviso.
# (Si prefieres las barras bonitas: `uv add --dev ipywidgets` y quita esta línea.)
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

from agente.config import ARQUITECTURAS, MODELO, REPETICIONES
from agente.corpus import dir_corpus, cargar_xbrl, raiz_repo
from agente.interfaz import (responder, ejecutar, puntuar, comparar, resumir, calentar,
                             dir_rep, dir_arquitectura, dir_resultados, leer_golden)
from agente.resultado import pretty_trace

# --- lo que vamos a evaluar en esta pasada ---
ARQ    = "baseline"                 # baseline · a1_guardrails · a2_retrieval · a3_hibrido · a4_comparativas
REPS   = REPETICIONES               # 3
GOLDEN = "data/golden_set.jsonl"

# --- comprobaciones, sin gastar nada ---
print("raíz del repo :", raiz_repo())
print("corpus        :", dir_corpus(), "·", len(cargar_xbrl()), "hechos XBRL")
print("modelo        :", MODELO)
print("clave OpenRouter en el entorno:", bool(os.environ.get("OPENROUTER_API_KEY")))   # solo sí/no, nunca el valor
HAY_CLAVE = bool(os.environ.get("OPENROUTER_API_KEY"))

golden = leer_golden(GOLDEN)
print(f"golden set    : {len(golden)} preguntas · "
      f"{sum(1 for g in golden if g.get('ancla_texto'))} con ancla")

# --- calentamiento: fuera del cronómetro, una sola vez por kernel ---
print("\ncalentando el retrieval…")
calentar()
print("HF_HUB_OFFLINE =", os.environ.get("HF_HUB_OFFLINE", "(sin poner)"))

# Las arquitecturas, como tabla: qué enciende cada una
display(pd.DataFrame([a.como_dict() for n, a in ARQUITECTURAS.items() if n != "final"])
        .set_index("nombre")[["limites","verificador_cifras","verificador_cita","esquema_estricto",
                              "prompt","filtros_forzados","reescritura","hibrido","descripcion"]])

### Limpiar antes de empezar

`ejecutar()` es **idempotente**: si ya existe `crudo/<id>.json`, salta la pregunta. Eso permite relanzar una ejecución cortada sin repetir lo hecho — pero también significa que **una medición mala se queda guardada** hasta que la borres.

- `BORRAR_TODO = True` deja la arquitectura a cero (crudo, tablas y resumen).
- `BORRAR_REP = n` borra solo esa repetición.
- Con los dos apagados, la celda **solo informa** de lo que hay.

Acuérdate de volver a dejarlos apagados después de borrar, o la próxima vez que pases por aquí te llevas por delante lo bueno.

In [ ]:
BORRAR_REP  = None           # p. ej. 1 para rehacer solo la repetición 1 de ARQ
BORRAR_TODO = False          # True para dejar ARQ completamente a cero

import shutil

if BORRAR_TODO:
    carpeta = dir_arquitectura(ARQ)
    if carpeta.is_dir():
        shutil.rmtree(carpeta)
    print(f"borrada entera: {carpeta}")
elif BORRAR_REP is not None:
    carpeta = dir_rep(ARQ, BORRAR_REP)
    if carpeta.is_dir():
        shutil.rmtree(carpeta)
    print(f"borrada: {carpeta}")
else:
    for carpeta in sorted(dir_arquitectura(ARQ).glob("rep*")):
        n = len(list((carpeta / "crudo").glob("*.json"))) if (carpeta / "crudo").is_dir() else 0
        print(f"{carpeta.name}: {n} preguntas guardadas")
    print("(nada borrado)")

## 2 · Una pregunta suelta

Antes de lanzar 60 llamadas, una. Se ve la trayectoria entera: qué herramienta pidió, con qué argumentos, qué le devolvió, y la respuesta estructurada. Es la misma `pretty_trace` de clase.

In [ ]:
PREGUNTA = "¿Cuál fue el revenue de NVIDIA en FY2025?"

if HAY_CLAVE:
    r = responder(PREGUNTA, thread_id="suelta", arquitectura=ARQ)
    print(pretty_trace(r))
    print(f"\n  [{r['latencia_s']:.1f} s · {r['coste_usd']*100:.2f} ¢]")
else:
    print("Sin clave: este bloque no se ejecuta.")

## 3 · Ejecutar una arquitectura, `REPS` veces

`ejecutar()` guarda **un JSON por pregunta** en `resultados/agente/<ARQ>/rep<n>/crudo/` con la trayectoria completa. Si esto se corta a mitad, vuelve a ejecutar la celda y sigue donde estaba.

`puntuar()` no gasta API: lee esos JSON, aplica los tres evaluadores y escribe `tabla.csv`. Como está separado, si mañana corregimos un evaluador se re-puntúa todo en segundos.

El `thread_id` lleva la repetición dentro (`baseline-rep2-gX-013`): sin eso, la repetición 2 vería la conversación de la 1 en el checkpointer.

In [ ]:
tablas = {}
if HAY_CLAVE:
    for rep in range(1, REPS + 1):
        print(f"\n== {ARQ} · repetición {rep}/{REPS} ==")
        ejecutar(GOLDEN, ARQ, rep)                 # API · idempotente · calienta antes del bucle
        tablas[rep] = puntuar(ARQ, rep)            # sin API · incluye recall@5
        display(resumir(tablas[rep], f"{ARQ} rep{rep}"))
else:
    # Sin clave se puede puntuar lo que ya esté guardado de otras veces.
    for carpeta in sorted(dir_arquitectura(ARQ).glob("rep*")):
        if (carpeta / "crudo").is_dir() and any((carpeta / "crudo").glob("*.json")):
            tablas[int(carpeta.name[3:])] = puntuar(ARQ, int(carpeta.name[3:]), con_recall=False)
    print("Sin clave: puntuado lo guardado →", list(tablas) or "nada")

## 4 · Leer la tabla: qué falla y por qué

Una fila por pregunta. Las columnas que importan:

- **`acierto`**: todos los evaluadores aplicables en `True`.
- **`cita` / `cifra` / `trayectoria`**: los tres del enunciado. Vacío = no aplica a esa familia.
- **`herramientas`**: el camino que siguió. Una numérica sin `get_xbrl_fact` suspende trayectoria aunque la cifra sea correcta.
- **`fuente`**: si dice `ninguna` en una pregunta con respuesta, no la encontró; si dice `xbrl` con una cifra que no cuadra, la leyó de donde no debía o la inventó.
- **`recall5` / `pos_ancla`**: el retriever, con los filtros del golden. `pos_ancla=6` y `pos_ancla=900` fallan igual el recall@5 y no son el mismo problema.

In [ ]:
REP_A_MIRAR = 1
if REP_A_MIRAR in tablas:
    t = tablas[REP_A_MIRAR]

    # `cifra`/`cita`/`trayectoria` son los VEREDICTOS; `cifra_dada`/`cita_dada`,
    # lo que respondió el agente. Ver los dos juntos es lo que delata si un fallo
    # es de capacidad o de convención (p. ej. poner la variación donde se espera
    # el valor del ejercicio).
    cols = ["id","familia","acierto","cita","cifra","trayectoria","fuente",
            "cifra_dada","cifra_esperada","ratio_cifra","herramientas",
            "n_llamadas","coste_usd","latencia_s","recall5","pos_ancla"]
    cols = [c for c in cols if c in t]

    def colorear(fila):
        return ["background-color:#fde2e2" if fila.get("acierto") is False else
                ("background-color:#e2f5e2" if fila.get("acierto") is True else "") for _ in fila]

    display(t[cols].style.apply(colorear, axis=1)
            .format({"coste_usd": "{:.4f}", "latencia_s": "{:.1f}",
                     "cifra_dada": "{:,.0f}", "cifra_esperada": "{:,.0f}",
                     "ratio_cifra": "{:.3f}"}, na_rep="—"))

    fallos = t[t["acierto"] == False]                     # noqa: E712
    print(f"\n{len(fallos)} fallos de {len(t)}. Por familia:")
    display(t.groupby("familia")["acierto"].agg(["mean", "count"]).rename(columns={"mean": "tasa"}))

    # Instrumentación: qué guardrail actuó. Un guardrail que nunca salta no ha
    # aportado nada, y eso se cuenta en vez de estimarse con una ablación.
    instr = [c for c in ["corrigio_cifra","corrigio_cita","reintentos_esquema",
                         "limite_alcanzado","n_busquedas","busquedas_con_ticker",
                         "busquedas_con_item","uso_read_section"] if c in t]
    if instr:
        print("\nGuardrails y retrieval (media sobre las 20 preguntas):")
        display(t[instr].mean().to_frame("valor").T)

    print("\nLas 3 más lentas y las 3 más caras:")
    display(t.nlargest(3, "latencia_s")[["id","latencia_s","n_llamadas","herramientas"]])
    display(t.nlargest(3, "coste_usd")[["id","coste_usd","n_llamadas","herramientas"]])
else:
    print("No hay tabla para esa repetición.")

### Las trayectorias de los fallos

Esto es lo que en clase había que imprimir a mano. Aquí se lee de lo guardado: **no gasta nada**. Para cada fallo, la pregunta, lo esperado, la trayectoria y la respuesta.

In [ ]:
def ver_traza(arq, rep, id_):
    reg = json.loads((dir_rep(arq, rep) / "crudo" / f"{id_}.json").read_text(encoding="utf-8"))
    it = reg["item"]
    print("=" * 88)
    print(f"[{it['id']} · {it['familia']}] {it['pregunta']}")
    print(f"  esperado: {str(it.get('respuesta_esperada'))[:110]}")
    if it.get("herramienta_esperada"):
        print(f"  herramientas esperadas: {it['herramienta_esperada']}")
    print()
    print(reg.get("error") or pretty_trace(reg["resultado"]))

if REP_A_MIRAR in tablas:
    for id_ in tablas[REP_A_MIRAR].loc[tablas[REP_A_MIRAR]["acierto"] == False, "id"]:    # noqa: E712
        ver_traza(ARQ, REP_A_MIRAR, id_)

# Y cualquier otra, aunque haya acertado:
#   ver_traza(ARQ, 1, "gX-003")

## 5 · La comparativa: una fila por arquitectura

Media de las repeticiones, con mínimo y máximo para ver si un salto es mejora o ruido. `comparativa.md` es la tabla del informe con el mejor valor de cada columna en negrita.

In [ ]:
comp = comparar()
cols = ["arquitectura","reps","acierto","acierto numerica","acierto extractiva","acierto comparativa",
        "cita","cifra","trayectoria","recall@5","coste medio (¢)","latencia media (s)","llamadas/pregunta"]
display(comp[[c for c in cols if c in comp]].round(3))

if len(comp) and "acierto min" in comp:
    print("\nRango entre repeticiones (acierto):")
    display(comp[["arquitectura","acierto min","acierto","acierto max"]].round(3))

md_path = dir_resultados() / "comparativa.md"
if md_path.is_file():
    display(Markdown(md_path.read_text(encoding="utf-8")))

## 6 · El retrieval solo: recall@5 por configuración

La tabla del §4.4. No interviene el agente: se mide el buscador con los filtros del golden set. Cero llamadas de API salvo la reescritura de la consulta (una por pregunta, cacheada en `resultados/retrieval/reescrituras.json`).

`posiciones` dice en qué puesto quedó el ancla en cada configuración: es lo que explica **por qué** una configuración gana.

In [ ]:
from agente.recall import medir_recall, CONFIGS

configs = dict(CONFIGS)
if not HAY_CLAVE:                       # sin clave no se puede reescribir
    configs = {k: v for k, v in configs.items() if "reescritura" not in k}

recall = medir_recall(configs)
display(recall)

pos = pd.read_csv(dir_resultados() / "retrieval" / "posiciones.csv")
display(pos.pivot(index="id", columns="config", values="pos_ancla"))

## Siguiente peldaño

1. Cambia `ARQ` en el bloque 1: `a1_guardrails` → `a2_retrieval` → `a3_hibrido` → `a4_comparativas`.
2. Ejecuta los bloques 3, 4 y 5.
3. La comparativa crece una fila. Lo que empeore, también se cuenta.

Y al congelar el baseline, en la terminal:

```
git tag baseline-congelado
git add resultados && git commit -m "Baseline congelado: 3 repeticiones"
```